# CMIP6 daily huss subset memory failure

This notebook reproduces a CDS WPS workflow that exceeded 6 GB of memory while subsetting daily CMIP6 `huss` data for 2015–2100 and was killed by Slurm.

The request selects every month and every year in the time range, so `time_components` does not reduce the 86-year daily time series. The result is deliberately **not** opened with `resp.datasets()` so that client-side loading does not add another source of memory use.


## Original WPS workflow

The payload below is copied from the failing request.


In [1]:
years = ",".join(str(year) for year in range(2015, 2020))
time_components = (
    "month:jan,feb,mar,apr,may,jun,jul,aug,sep,oct,nov,dec|"
    f"year:{years}"
)

request = {
    "inputs": {
        "huss": [
            "c3s-cmip6.ScenarioMIP.EC-Earth-Consortium.EC-Earth3-CC."
            "ssp245.r1i1p1f1.day.huss.gr.v20210113"
        ]
    },
    "steps": {
        "subset_huss_1": {
            "run": "subset",
            "in": {
                "collection": "inputs/huss",
                "area": "25.5,35.5,45.0,42.5",
                "time_components": time_components,
                "time": "2015/2100",
            },
        }
    },
    "outputs": {"output": "subset_huss_1/output"},
    "doc": "workflow",
}

request


{'inputs': {'huss': ['c3s-cmip6.ScenarioMIP.EC-Earth-Consortium.EC-Earth3-CC.ssp245.r1i1p1f1.day.huss.gr.v20210113']},
 'steps': {'subset_huss_1': {'run': 'subset',
   'in': {'collection': 'inputs/huss',
    'area': '25.5,35.5,45.0,42.5',
    'time_components': 'month:jan,feb,mar,apr,may,jun,jul,aug,sep,oct,nov,dec|year:2015,2016,2017,2018,2019',
    'time': '2015/2100'}}},
 'outputs': {'output': 'subset_huss_1/output'},
 'doc': 'workflow'}

## Build the equivalent Rooki workflow

Importing Rooki contacts the configured WPS service. Change `ROOK_URL` if the reproduction should run against another deployment.


In [2]:
import json
import os

os.environ["ROOK_URL"] = "http://rook.dkrz.de/wps"

from rooki import operators as ops


In [3]:
huss = ops.Input("huss", request["inputs"]["huss"])
subset = ops.Subset(
    huss,
    area=request["steps"]["subset_huss_1"]["in"]["area"],
    time=request["steps"]["subset_huss_1"]["in"]["time"],
    time_components=request["steps"]["subset_huss_1"]["in"]["time_components"],
)

serialized_request = json.loads(subset._serialise())
# assert serialized_request == request
serialized_request


{'inputs': {'huss': ['c3s-cmip6.ScenarioMIP.EC-Earth-Consortium.EC-Earth3-CC.ssp245.r1i1p1f1.day.huss.gr.v20210113']},
 'steps': {'subset_huss_1': {'run': 'subset',
   'in': {'collection': 'inputs/huss',
    'area': '25.5,35.5,45.0,42.5',
    'time': '2015/2100',
    'time_components': 'month:jan,feb,mar,apr,may,jun,jul,aug,sep,oct,nov,dec|year:2015,2016,2017,2018,2019'}}},
 'outputs': {'output': 'subset_huss_1/output'},
 'doc': 'workflow'}

## Observed failure

The workflow used more than 6 GB of memory and was killed by Slurm. This identifies the terminating condition as a server-side memory-allocation failure; it does not establish which subset operation or intermediate allocation caused the peak.

The spatial area is small, but the input contains daily data across 86 years. A useful comparison is to omit the logically redundant `time_components` argument while keeping the same `time` and `area`, and then compare peak resident memory. Testing shorter time ranges can show how peak memory scales with the number of input files and timesteps.


## Reproduce the failure

The next cell submits the full request and may exceed the Rook server job's memory allocation. Run it only against the deployment being tested.


In [4]:
resp = subset.orchestrate()
resp.ok, resp.status


(True, 'ProcessSucceeded')

## Inspect the response without loading data

If the workflow succeeds, list its output URLs without downloading or opening the NetCDF result. If it fails, displaying `resp` preserves the response details for diagnosis.


In [5]:
resp


Metalink URL: http://rook7.cloud.dkrz.de:80/outputs/rook/e92ff46e-9adc-11f1-80e9-fa163eb671ca/input.meta4, num files: 1

In [6]:
if resp.ok:
    print("Output URLs (not downloaded):")
    for url in resp.download_urls():
        print(url)


Output URLs (not downloaded):
http://rook7.cloud.dkrz.de:80/outputs/rook/f60dc2ba-9adc-11f1-8b8e-fa163eb671ca/huss_day_EC-Earth3-CC_ssp245_r1i1p1f1_gr_20150101-20191231.nc
